# 03 - ELO Prediction Model

Fit a regression model that predicts a player's ELO from their Average Centipawn Loss (ACPL).

Uses the 200k sampled Lichess games with real Stockfish evals to calibrate:
1. Compute ACPL per player per game from the moves CSV (real Stockfish centipawns)
2. Join with known ELO from the games CSV
3. Fit `ELO = a - b × ln(ACPL)` using nonlinear regression
4. Also fit a multivariate model (ACPL + blunder rate + captures + checks)
5. PCA player profile clustering
6. Export coefficients to `elo_model.json` for the React frontend

## Setup

In [ ]:
import subprocess, sys

def install(pkg):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

install("scikit-learn")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.optimize import curve_fit
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
import json
import os

plt.rcParams['figure.dpi'] = 100
plt.rcParams['figure.figsize'] = (12, 6)
sns.set_style('whitegrid')

print('Setup done.')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DATA_DIR = '/content/drive/MyDrive/Learning-document/Data Visualization/Project 2/data'
FRONTEND_DIR = os.path.join(DATA_DIR, 'frontend')
os.makedirs(FRONTEND_DIR, exist_ok=True)

games = pd.read_csv(os.path.join(DATA_DIR, 'lichess_sampled_games.csv'))
moves = pd.read_csv(os.path.join(DATA_DIR, 'lichess_sampled_moves.csv'))

print(f'Games: {len(games):,}')
print(f'Moves (eval): {len(moves):,}')
print(f'\nMoves columns: {list(moves.columns)}')
print(f'Games columns: {list(games.columns)}')